# Quick conversions

Convert single colors, CSS strings and arrays between profiles and color spaces with lcms2.convert.

In [1]:
import numpy as np

import lcms2

`lcms2.convert(data, source, target)` takes profile or color space names. A list of integers is read as
8-bit RGB; the result is float for Lab, XYZ and the analytic spaces.

In [2]:
red = lcms2.convert([255, 0, 0], "sRGB", "Lab", intent="relative")
print(red.round(2))
print(lcms2.convert([1.0, 0.0, 0.0], "sRGB", "oklab").round(4))

[54.29 80.81 69.89]
[0.6279 0.2249 0.1258]


Colors can be CSS strings. `parse_color` returns floats in 0..1 and `to_hex` goes back.

In [3]:
rgb = lcms2.parse_color("rebeccapurple")
print(rgb.round(3), lcms2.to_hex(rgb))
print(lcms2.convert(rgb, "sRGB", "Display P3").round(4))

[0.4 0.2 0.6] #663399
[0.3737 0.2103 0.5791]


Arrays keep their shape: the last axis is the channels. The input dtype picks the pixel format and the
output keeps it, so uint8 in gives uint8 out.

In [4]:
rng = np.random.default_rng(0)
img = rng.integers(0, 256, (2, 3, 3), dtype=np.uint8)
p3 = lcms2.convert(img, "sRGB", "Display P3")
print(img.dtype, "->", p3.dtype, p3.shape)
print(p3[0, 0])

uint8 -> uint8 (2, 3, 3)
[102 129 189]


`out_dtype` asks for another type. Integer input converted to Lab or XYZ comes back as float64 on its own.

In [5]:
print(lcms2.convert(img, "sRGB", "Display P3", out_dtype=np.float32)[0, 0].round(4))

[0.4013 0.506  0.7415]


In [6]:
lab = lcms2.convert(img, "sRGB", "Lab")
print(lab.dtype)
print(lab[0, 0].round(2))

float64
[ 53.83   1.26 -37.81]


Float and 16-bit data work the same way. Here 16-bit values go to linear light through a color space.

In [7]:
img16 = img.astype(np.uint16) * 257
print(lcms2.convert(img16, "sRGB", "srgb-linear")[0, 0].round(4))

[0.1145 0.2232 0.5395]


Names are case-insensitive. `Profile.builtins()` lists the profiles and `ColorSpace.list()` the spaces.

In [8]:
print(lcms2.Profile.builtins()[:6])
print(lcms2.spaces.ColorSpace.list()[:6])
print(lcms2.get_version())

['sRGB', 'Lab', 'Lab v2', 'XYZ', 'NULL', 'Adobe RGB (1998)']
['a98-rgb', 'cmyk', 'display-p3', 'display-p3-linear', 'gray', 'hsv']
2.20
